# Reasoning Memory: Colab
Включите GPU runtime. Первая ячейка клонирует или обновляет `/content/reasoning-memory` через fast-forward.
Для приватного репозитория добавьте в Colab Secrets `GITHUB_TOKEN` с правом **Contents: Read-only** и разрешите доступ ноутбуку.
Без секрета появится скрытый ввод токена. Повторный запуск сохраняет папки `runs/`.


In [ ]:
from pathlib import Path
import json
import os
import subprocess
import sys
import tempfile
from getpass import getpass
from google.colab import files, userdata

REPO_URL = 'https://github.com/Ferraronp/reasoning-memory.git'
BRANCH = 'main'
PROJECT_ROOT = Path('/content/reasoning-memory')

def sync_project(repo_url, branch, project_root, git_env=None):
    project_root = Path(project_root)
    def git(*args, cwd=None):
        result = subprocess.run(['git', '-c', 'credential.helper=', *args], cwd=cwd,
                                env=git_env, capture_output=True, text=True)
        if result.returncode:
            raise RuntimeError(result.stderr.strip() or result.stdout.strip() or 'Git failed')
        return result.stdout.strip()
    if not project_root.exists():
        git('clone', '--branch', branch, '--single-branch', repo_url, str(project_root))
    else:
        if not (project_root / '.git').is_dir():
            raise RuntimeError(f'{project_root} exists but is not a git clone. Choose another PROJECT_ROOT.')
        origin = git('remote', 'get-url', 'origin', cwd=project_root)
        if origin.rstrip('/') != repo_url.rstrip('/'):
            raise RuntimeError('The existing folder has a different origin; choose another PROJECT_ROOT.')
        if git('branch', '--show-current', cwd=project_root) != branch:
            raise RuntimeError(f'The checkout is not on {branch}. Switch it manually before updating.')
        if git('status', '--porcelain', '--untracked-files=no', cwd=project_root):
            raise RuntimeError('Local source changes found. Commit/stash them before updating; nothing was overwritten.')
        git('fetch', 'origin', branch, cwd=project_root)
        git('merge', '--ff-only', 'FETCH_HEAD', cwd=project_root)
    print('Project:', project_root)
    print('Commit:', git('log', '-1', '--format=%h %s', cwd=project_root))

def sync_private_project():
    try:
        token = userdata.get('GITHUB_TOKEN')
    except Exception:
        token = None
    if not token:
        token = getpass('GitHub token (read-only Contents for this repository): ').strip()
    if not token:
        raise ValueError('A token is required for this private repository')
    # Helper contains only environment-variable references, never the actual token.
    with tempfile.TemporaryDirectory(prefix='rm-git-auth-') as auth_dir:
        askpass = Path(auth_dir) / 'askpass.sh'
        askpass.write_text('#!/bin/sh\ncase "$1" in\n  *Username*) printf "%s\\n" "x-access-token" ;;\n  *) printf "%s\\n" "$RM_GITHUB_TOKEN" ;;\nesac\n')
        askpass.chmod(0o700)
        env = os.environ.copy()
        env.update(GIT_ASKPASS=str(askpass), GIT_TERMINAL_PROMPT='0', RM_GITHUB_TOKEN=token)
        try:
            sync_project(REPO_URL, BRANCH, PROJECT_ROOT, env)
        finally:
            env.pop('RM_GITHUB_TOKEN', None)
            token = None

sync_private_project()
assert (PROJECT_ROOT / 'pyproject.toml').is_file()


## Установка и краткая проверка
`doctor` покажет версию, коммит и GPU. Полный список библиотек сохраняется в `runs/.../environment.json`.


In [ ]:
def run_process(argv):
    # Pipes are read in Python so Jupyter/Colab reliably displays child output.
    with subprocess.Popen(argv, cwd=PROJECT_ROOT, stdout=subprocess.PIPE,
                          stderr=subprocess.STDOUT, text=True, bufsize=1) as process:
        for line in process.stdout:
            print(line, end='', flush=True)
        return process.wait()

if run_process([sys.executable, '-m', 'pip', 'install', '--quiet', '-e', f'{PROJECT_ROOT}[hf]']) != 0:
    raise RuntimeError('Installation failed; see output above')

def run_cli(*args):
    code = run_process([sys.executable, '-u', '-m', 'reasoning_memory', *map(str, args)])
    if code not in (0, 2):
        raise RuntimeError(f'Ошибка запуска, exit={code}; см. вывод выше')
    if code == 2:
        print('Запуск закончен с диагностическими ошибками. Причина и текст генерации показаны выше.')
    # Intentionally no CompletedProcess return value.

run_cli('doctor')


## Два этапа на Qwen3-1.7B
Сначала модель находит `r=5`. После общего вывода создаются `full` и `compact`; второе пользовательское сообщение просит вычислить `4*r-3`.
Начните с `LIMIT = 1` (ожидается `17`), затем попробуйте `3` (ответы `17`, `25`, `53`).
Этот режим использует два сообщения. Итоговый ответ модели сохраняется как вывод второго этапа.
Проверка формата строгая: `Conclusion: 17` не равно `17`.


In [ ]:
LIMIT = 1  # Затем 3 — все три задачи
run_cli('pair', '--config', 'configs/qwen3_17b_fp16_chat_two_stage.json',
        '--tasks', 'data/two_stage.jsonl', '--limit', LIMIT)


## Результаты последнего запуска
Смотрите завершение веток и ответы в `results.json` и `summary.json`. Подробности каждой генерации — в `task_00000/events.jsonl`.


In [ ]:
latest = max((PROJECT_ROOT / 'runs').iterdir(), key=lambda p: p.name)
print('Run:', latest)
for filename in ['manifest.json', 'results.json', 'summary.json']:
    path = latest / filename
    if path.exists():
        print(filename, path.read_text()[:12000])
# Полные входы и выходы: latest / 'task_00000/events.jsonl'


## Скачать результаты
Диск Colab временный. Архив включает все запуски из `runs/`.


In [ ]:
import shutil
result_zip = shutil.make_archive('/content/reasoning-memory-results', 'zip', PROJECT_ROOT, 'runs')
files.download(result_zip)
